[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shriramnag/NeonVido-AI/blob/main/colab/NeonVido_Server.ipynb)

# 🎬 NeonVido-AI (v1.0.004) — Shriramnag All-in-One HF Models + Anti-Disconnect
**निर्देश:** `Runtime > Change runtime type > T4 GPU` चुनें और Play (▶) दबाएं।

In [ ]:
# 1. Colab ब्राउज़र टैब को सोने से रोकने के लिए Anti-Disconnect JavaScript चालू करें
from IPython.display import display, Javascript
display(Javascript('''
function KeepColabAlive(){
    console.log("NeonVido-AI Keep-Alive Ping...");
    const btn = document.querySelector("colab-connect-button") || document.querySelector("#top-toolbar");
    if(btn) { btn.dispatchEvent(new MouseEvent('mousemove', {bubbles: true})); }
}
setInterval(KeepColabAlive, 45000);
'''))

# 2. ज़रूरी पैकेज इंस्टॉल करें (PyTorch को बिना छेड़े)
!pip install -q diffusers transformers accelerate bitsandbytes fastapi uvicorn nest-asyncio imageio[ffmpeg] huggingface_hub opencv-python-headless peft soundfile scipy
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import os, re, gc, time, threading, subprocess, torch
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel
from diffusers import LTXPipeline
from diffusers.utils import export_to_video
from huggingface_hub import hf_hub_download, snapshot_download
import uvicorn

gc.collect()
torch.cuda.empty_cache()

# 3. आपके Hugging Face रिपॉजिटरी (Shriramnag/Shiv_AI_video_model) से कंप्रेस्ड मॉडल लोड करें
print("⏳ 1/3: आपका Shiv_AI_video_model डाउनलोड व कंप्रेस हो रहा है...")
local_model_dir = snapshot_download(
    repo_id="Shriramnag/Shiv_AI_video_model",
    allow_patterns=["*.json", "*.safetensors", "*.pth", "scheduler/*", "text_encoder/*", "tokenizer/*", "vae/*", "tts_config/*"]
)

print("⏳ 2/3: आपका Shiv-AI-Motion-Enhancer (8-Step) जुड़ रहा है...")
try:
    motion_lora = hf_hub_download(
        repo_id="Shriramnag/Shiv-AI-Motion-Enhancer",
        filename="minimax-i2v_8Step_model.safetensors"
    )
except:
    motion_lora = None

print("⏳ 3/3: लो-मेमोरी (bfloat16) में वीडियो इंजन तैयार हो रहा है...")
pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print("✅ सभी मॉडल्स कंप्रेस होकर लोड हो गए हैं! (Wav2Lip और Hindi TTS फाइलें भी तैयार हैं)")

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

last_heartbeat = time.time()

@app.get("/ping")
async def ping():
    global last_heartbeat
    last_heartbeat = time.time()
    return {"status": "alive", "gpu": torch.cuda.get_device_name(0)}

class VideoReq(BaseModel):
    model: str
    prompt: str
    target_minutes: float
    ratio: str
    extend: bool = False
    use_voice: bool = False
    voice_script: str = ""

@app.post("/generate")
async def generate(req: VideoReq):
    w, h = (480, 704) if req.ratio == "9:16" else (704, 480)
    base_video = "/content/current_master.mp4"

    loops = 1 if req.target_minutes <= 0.2 else min(int((req.target_minutes * 60) // 5), 120)
    if req.extend and os.path.exists(base_video):
        loops = 1

    clip_files = []
    if req.extend and os.path.exists(base_video):
        clip_files.append(base_video)

    prompts_list = [p.strip() for p in req.prompt.split("\n") if p.strip()]

    for i in range(loops):
        current_prompt = prompts_list[i % len(prompts_list)]
        print(f"🎬 क्लिप {i+1}/{loops} बन रही है: {current_prompt}")
        video_frames = pipe(
            prompt=current_prompt,
            negative_prompt="worst quality, inconsistent motion, blurry, jittery, distorted",
            width=w,
            height=h,
            num_frames=81,
            num_inference_steps=20
        ).frames[0]
        part_path = f"/content/part_{i}.mp4"
        export_to_video(video_frames, part_path, fps=24)
        clip_files.append(part_path)
        gc.collect()
        torch.cuda.empty_cache()

    with open("/content/list.txt", "w") as f:
        for c in clip_files:
            f.write(f"file '{c}'\n")

    subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", "/content/list.txt", "-c", "copy", base_video], check=True)
    print("✅ फाइनल वीडियो तैयार है!")
    return FileResponse(base_video, media_type="video/mp4", filename="NeonVido-AI.mp4")

def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=run_api, daemon=True).start()
time.sleep(2)

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n🚀 आपका NEONVIDO-AI LIVE URL:\n" + m.group(0) + "\n")
        break